In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# Case Study: Building a Hybrid Memory System for an AI Clinical Assistant
## Implementation Notebook

Welcome to the implementation notebook for the MedFlow AI hybrid memory case study. In this notebook, you will build a complete multi-session memory system for a clinical AI assistant, progressing from data exploration through a fully evaluated hybrid memory architecture.

**What you will build:**
- A sliding window baseline for clinical conversation memory
- A vector-backed long-term memory system using FAISS and sentence-transformers
- A structured entity store for safety-critical clinical facts (allergies, medications, diagnoses)
- A running summary system that compresses patient history
- A hybrid memory orchestrator that scores and ranks memories from all subsystems
- A complete evaluation framework comparing your hybrid system against the baseline

**Prerequisites:** Familiarity with Python, NumPy, and basic NLP concepts. Understanding of the Memory Architectures article (buffer, sliding window, summary, entity, vector, and hybrid memory) is essential.

**Estimated time:** 3-4 hours

---

## 3.1 Data Acquisition and Preprocessing

In this section, you will set up the environment and load a clinical conversation dataset. We use a combination of content from MedQuAD (a real medical QA dataset from NIH/NLM sources) and synthetic multi-session patient conversations that simulate realistic follow-up care scenarios.

The synthetic conversations are carefully constructed to contain cross-session information dependencies -- facts stated in early sessions that are clinically critical in later sessions. This lets us rigorously test whether the memory system can retrieve the right information at the right time.

In [ ]:
# Install dependencies
!pip install -q sentence-transformers faiss-cpu numpy pandas matplotlib seaborn tqdm scikit-learn

import json
import re
import time
import math
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import deque, defaultdict
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict, Any
from tqdm.notebook import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)

print("All dependencies loaded successfully.")

In [ ]:
# --- Clinical Knowledge Base ---
# Real medical content sourced from publicly available medical references
# (MedQuAD-style question-answer pairs covering cardiology, endocrinology, oncology)

MEDICATIONS = {
    "metformin": {"class": "biguanide", "use": "Type 2 Diabetes", "common_doses": ["500mg", "850mg", "1000mg"],
                  "side_effects": ["GI upset", "nausea", "diarrhea", "lactic acidosis"],
                  "contraindications": ["renal impairment", "hepatic disease"]},
    "lisinopril": {"class": "ACE inhibitor", "use": "hypertension", "common_doses": ["5mg", "10mg", "20mg"],
                   "side_effects": ["dry cough", "dizziness", "hyperkalemia"],
                   "contraindications": ["pregnancy", "bilateral renal artery stenosis"]},
    "atorvastatin": {"class": "statin", "use": "hyperlipidemia", "common_doses": ["10mg", "20mg", "40mg", "80mg"],
                     "side_effects": ["myalgia", "elevated liver enzymes", "rhabdomyolysis"],
                     "contraindications": ["active liver disease", "pregnancy"]},
    "metoprolol": {"class": "beta-blocker", "use": "heart failure, hypertension", "common_doses": ["25mg", "50mg", "100mg"],
                   "side_effects": ["bradycardia", "fatigue", "cold extremities"],
                   "contraindications": ["severe bradycardia", "decompensated heart failure"]},
    "amlodipine": {"class": "calcium channel blocker", "use": "hypertension", "common_doses": ["2.5mg", "5mg", "10mg"],
                   "side_effects": ["peripheral edema", "dizziness", "flushing"],
                   "contraindications": ["severe aortic stenosis"]},
    "levothyroxine": {"class": "thyroid hormone", "use": "hypothyroidism", "common_doses": ["25mcg", "50mcg", "100mcg"],
                      "side_effects": ["palpitations", "tremor", "weight loss"],
                      "contraindications": ["thyrotoxicosis", "uncorrected adrenal insufficiency"]},
    "insulin glargine": {"class": "long-acting insulin", "use": "diabetes", "common_doses": ["10 units", "20 units", "30 units"],
                         "side_effects": ["hypoglycemia", "weight gain", "injection site reactions"],
                         "contraindications": ["hypoglycemia"]},
    "warfarin": {"class": "anticoagulant", "use": "atrial fibrillation, DVT", "common_doses": ["2mg", "5mg", "7.5mg"],
                 "side_effects": ["bleeding", "bruising", "skin necrosis"],
                 "contraindications": ["active bleeding", "pregnancy"]},
}

ALLERGIES_DB = {
    "penicillin": {"cross_reactive": ["amoxicillin", "ampicillin"], "severity": "severe", "reaction": "anaphylaxis"},
    "sulfonamides": {"cross_reactive": ["trimethoprim-sulfamethoxazole", "sulfasalazine"], "severity": "severe", "reaction": "rash, Stevens-Johnson syndrome"},
    "NSAIDs": {"cross_reactive": ["ibuprofen", "naproxen", "aspirin"], "severity": "moderate", "reaction": "GI bleeding, bronchospasm"},
    "codeine": {"cross_reactive": ["morphine", "hydrocodone"], "severity": "moderate", "reaction": "nausea, respiratory depression"},
    "latex": {"cross_reactive": [], "severity": "moderate", "reaction": "contact dermatitis, anaphylaxis"},
    "contrast dye": {"cross_reactive": ["iodinated contrast"], "severity": "severe", "reaction": "anaphylactoid reaction"},
}

DIAGNOSES = [
    "Type 2 Diabetes Mellitus", "Hypertension", "Hyperlipidemia",
    "Atrial Fibrillation", "Heart Failure (HFrEF)", "Hypothyroidism",
    "Chronic Kidney Disease Stage 3", "COPD", "Coronary Artery Disease",
    "Osteoarthritis", "GERD", "Depression", "Anxiety",
    "Breast Cancer (Stage IIA)", "Non-Small Cell Lung Cancer",
]

LAB_TESTS = {
    "A1C": {"unit": "%", "normal_range": (4.0, 5.6), "prediabetic": (5.7, 6.4), "diabetic_threshold": 6.5},
    "LDL": {"unit": "mg/dL", "normal_range": (0, 100), "borderline": (100, 130), "high_threshold": 130},
    "TSH": {"unit": "mIU/L", "normal_range": (0.4, 4.0), "hypo_threshold": 4.0, "hyper_threshold": 0.4},
    "creatinine": {"unit": "mg/dL", "normal_range": (0.6, 1.2), "elevated_threshold": 1.3},
    "eGFR": {"unit": "mL/min/1.73m2", "normal_range": (90, 120), "stage3_threshold": 60},
    "INR": {"unit": "", "normal_range": (0.8, 1.2), "therapeutic_range": (2.0, 3.0)},
    "BNP": {"unit": "pg/mL", "normal_range": (0, 100), "hf_threshold": 400},
}

print(f"Knowledge base loaded:")
print(f"  {len(MEDICATIONS)} medications")
print(f"  {len(ALLERGIES_DB)} allergy types")
print(f"  {len(DIAGNOSES)} diagnoses")
print(f"  {len(LAB_TESTS)} lab tests")

In [ ]:
# --- Synthetic Patient Generator ---
# Creates realistic multi-session patient records with cross-session dependencies

@dataclass
class PatientProfile:
    patient_id: str
    age: int
    sex: str
    diagnoses: List[str]
    medications: Dict[str, str]  # drug_name -> dosage
    allergies: List[str]
    lab_results: Dict[str, float]  # test_name -> value

@dataclass
class SessionMessage:
    role: str  # 'physician' or 'assistant'
    content: str
    timestamp: str

@dataclass
class PatientSession:
    session_id: int
    patient_id: str
    date: str
    messages: List[SessionMessage]
    entities_ground_truth: List[Dict]  # Ground truth for evaluation

def generate_patient_profiles(n_patients: int = 20) -> List[PatientProfile]:
    """Generate diverse patient profiles."""
    profiles = []
    for i in range(n_patients):
        n_diagnoses = random.randint(2, 5)
        selected_diagnoses = random.sample(DIAGNOSES, min(n_diagnoses, len(DIAGNOSES)))

        n_meds = random.randint(2, 6)
        selected_meds = random.sample(list(MEDICATIONS.keys()), min(n_meds, len(MEDICATIONS)))
        medications = {med: random.choice(MEDICATIONS[med]["common_doses"]) for med in selected_meds}

        n_allergies = random.randint(0, 3)
        selected_allergies = random.sample(list(ALLERGIES_DB.keys()), min(n_allergies, len(ALLERGIES_DB)))

        lab_results = {}
        for test_name, test_info in LAB_TESTS.items():
            low, high = test_info["normal_range"]
            # 40% chance of abnormal result
            if random.random() < 0.4:
                lab_results[test_name] = round(high * random.uniform(1.1, 1.8), 1)
            else:
                lab_results[test_name] = round(random.uniform(low, high), 1)

        profiles.append(PatientProfile(
            patient_id=f"PT-{i+1:04d}",
            age=random.randint(35, 80),
            sex=random.choice(["Male", "Female"]),
            diagnoses=selected_diagnoses,
            medications=medications,
            allergies=selected_allergies,
            lab_results=lab_results,
        ))
    return profiles


def generate_session(patient: PatientProfile, session_id: int,
                     base_date: str, prior_sessions: List[PatientSession]) -> PatientSession:
    """Generate a single clinical session with realistic dialogue.

    Session types by number:
    - Session 1: Initial visit (establishes baseline -- diagnoses, meds, allergies, labs)
    - Session 2-3: Follow-up visits (adjustments, new symptoms, lab reviews)
    - Session 4+: Ongoing management (cross-references prior sessions)
    """
    messages = []
    entities_gt = []

    if session_id == 1:
        # Initial visit: establish all baseline information
        # Allergies
        if patient.allergies:
            allergy_text = ", ".join(patient.allergies)
            messages.append(SessionMessage("physician",
                f"Let's start with allergies. Do you have any known drug allergies?",
                f"{base_date}T09:00:00"))
            messages.append(SessionMessage("assistant",
                f"Patient reports allergies to: {allergy_text}. "
                f"These have been documented and flagged as safety-critical.",
                f"{base_date}T09:01:00"))
            for a in patient.allergies:
                entities_gt.append({"type": "allergy", "name": a,
                    "value": ALLERGIES_DB[a]["reaction"], "importance": 1.0})

        # Medications
        med_lines = [f"{name} {dose}" for name, dose in patient.medications.items()]
        messages.append(SessionMessage("physician",
            f"What medications are you currently taking?",
            f"{base_date}T09:05:00"))
        messages.append(SessionMessage("assistant",
            f"Patient's current medication list: {'; '.join(med_lines)}. "
            f"All medications have been verified and documented.",
            f"{base_date}T09:06:00"))
        for name, dose in patient.medications.items():
            entities_gt.append({"type": "medication", "name": name,
                "value": dose, "importance": 0.5})

        # Diagnoses
        messages.append(SessionMessage("physician",
            f"Let me review your medical history. What conditions are you being treated for?",
            f"{base_date}T09:10:00"))
        messages.append(SessionMessage("assistant",
            f"Patient's active diagnoses include: {', '.join(patient.diagnoses)}. "
            f"Problem list has been updated accordingly.",
            f"{base_date}T09:11:00"))
        for d in patient.diagnoses:
            entities_gt.append({"type": "diagnosis", "name": d,
                "value": "active", "importance": 0.5})

        # Lab results
        lab_lines = [f"{name}: {val} {LAB_TESTS[name]['unit']}" for name, val in patient.lab_results.items()]
        messages.append(SessionMessage("physician",
            f"Let's review the latest lab work.",
            f"{base_date}T09:15:00"))
        messages.append(SessionMessage("assistant",
            f"Recent lab results: {'; '.join(lab_lines)}.",
            f"{base_date}T09:16:00"))
        for name, val in patient.lab_results.items():
            entities_gt.append({"type": "lab_result", "name": name,
                "value": str(val), "importance": 0.5})

    elif session_id == 2:
        # Follow-up: medication adjustment + new symptom
        med_to_adjust = random.choice(list(patient.medications.keys()))
        old_dose = patient.medications[med_to_adjust]
        new_dose = random.choice(MEDICATIONS[med_to_adjust]["common_doses"])
        while new_dose == old_dose:
            new_dose = random.choice(MEDICATIONS[med_to_adjust]["common_doses"])

        side_effect = random.choice(MEDICATIONS[med_to_adjust]["side_effects"])
        messages.append(SessionMessage("physician",
            f"How has the {med_to_adjust} been working since last visit?",
            f"{base_date}T09:00:00"))
        messages.append(SessionMessage("assistant",
            f"Patient reports experiencing {side_effect} since starting {med_to_adjust} {old_dose}.",
            f"{base_date}T09:01:00"))
        messages.append(SessionMessage("physician",
            f"Let's adjust the dose. Change {med_to_adjust} from {old_dose} to {new_dose}.",
            f"{base_date}T09:05:00"))
        messages.append(SessionMessage("assistant",
            f"Noted. {med_to_adjust.capitalize()} adjusted from {old_dose} to {new_dose} "
            f"due to {side_effect}. Will monitor at next visit.",
            f"{base_date}T09:06:00"))
        entities_gt.append({"type": "medication", "name": med_to_adjust,
            "value": f"{new_dose} (changed from {old_dose})", "importance": 0.5})

        # New lab result
        test = random.choice(list(patient.lab_results.keys()))
        new_val = round(patient.lab_results[test] * random.uniform(0.85, 1.15), 1)
        messages.append(SessionMessage("physician",
            f"I see we have new {test} results. What are they?",
            f"{base_date}T09:10:00"))
        messages.append(SessionMessage("assistant",
            f"{test} is now {new_val} {LAB_TESTS[test]['unit']}, "
            f"compared to {patient.lab_results[test]} {LAB_TESTS[test]['unit']} at last visit.",
            f"{base_date}T09:11:00"))
        entities_gt.append({"type": "lab_result", "name": test,
            "value": str(new_val), "importance": 0.5})

    elif session_id == 3:
        # Second follow-up: references back to session 1 and 2
        messages.append(SessionMessage("physician",
            f"Let's review the patient's overall trajectory since the initial visit.",
            f"{base_date}T09:00:00"))
        messages.append(SessionMessage("assistant",
            f"The patient was first seen on their initial visit with "
            f"{len(patient.diagnoses)} active diagnoses and {len(patient.medications)} medications. "
            f"Medication adjustments were made at the last follow-up visit.",
            f"{base_date}T09:01:00"))

        # Reference a specific prior decision
        if patient.allergies:
            messages.append(SessionMessage("physician",
                f"Patient has a skin infection. Can we use any penicillin-class antibiotics?",
                f"{base_date}T09:05:00"))
            if "penicillin" in patient.allergies:
                messages.append(SessionMessage("assistant",
                    f"WARNING: This patient has a documented severe allergy to penicillin "
                    f"from the initial visit. Penicillin-class antibiotics are contraindicated. "
                    f"Consider azithromycin or clindamycin as alternatives.",
                    f"{base_date}T09:06:00"))
            else:
                messages.append(SessionMessage("assistant",
                    f"No penicillin allergy documented. Amoxicillin would be appropriate.",
                    f"{base_date}T09:06:00"))

        # Discuss adding a new medication
        potential_new_meds = [m for m in MEDICATIONS if m not in patient.medications]
        if potential_new_meds:
            new_med = random.choice(potential_new_meds)
            new_dose = random.choice(MEDICATIONS[new_med]["common_doses"])
            messages.append(SessionMessage("physician",
                f"Based on the labs, I want to start {new_med} {new_dose}.",
                f"{base_date}T09:10:00"))
            messages.append(SessionMessage("assistant",
                f"Starting {new_med} {new_dose}. Checking for interactions with "
                f"current medications: {', '.join(patient.medications.keys())}. "
                f"No major interactions found. Added to medication list.",
                f"{base_date}T09:11:00"))
            entities_gt.append({"type": "medication", "name": new_med,
                "value": new_dose, "importance": 0.5})

    else:
        # Later sessions: more complex cross-references
        messages.append(SessionMessage("physician",
            f"This is a routine follow-up. Any new symptoms or concerns?",
            f"{base_date}T09:00:00"))
        messages.append(SessionMessage("assistant",
            f"Patient reports feeling generally well. No new acute complaints. "
            f"Continuing with current medication regimen.",
            f"{base_date}T09:01:00"))

        # Random lab check
        test = random.choice(list(patient.lab_results.keys()))
        new_val = round(patient.lab_results[test] * random.uniform(0.8, 1.2), 1)
        messages.append(SessionMessage("physician",
            f"Any updates on {test}?",
            f"{base_date}T09:05:00"))
        messages.append(SessionMessage("assistant",
            f"Latest {test}: {new_val} {LAB_TESTS[test]['unit']}.",
            f"{base_date}T09:06:00"))
        entities_gt.append({"type": "lab_result", "name": test,
            "value": str(new_val), "importance": 0.5})

    return PatientSession(
        session_id=session_id,
        patient_id=patient.patient_id,
        date=base_date,
        messages=messages,
        entities_ground_truth=entities_gt,
    )


# Generate dataset: 20 patients, 4 sessions each
patients = generate_patient_profiles(n_patients=20)
all_sessions = []

for patient in patients:
    dates = ["2024-01-15", "2024-02-01", "2024-03-01", "2024-04-15"]
    prior = []
    for sid in range(1, 5):
        session = generate_session(patient, sid, dates[sid-1], prior)
        all_sessions.append(session)
        prior.append(session)

print(f"Generated {len(patients)} patients with {len(all_sessions)} total sessions")
print(f"Average messages per session: {np.mean([len(s.messages) for s in all_sessions]):.1f}")
print(f"\nSample patient: {patients[0].patient_id}")
print(f"  Age: {patients[0].age}, Sex: {patients[0].sex}")
print(f"  Diagnoses: {patients[0].diagnoses}")
print(f"  Medications: {patients[0].medications}")
print(f"  Allergies: {patients[0].allergies}")

In [ ]:
# --- Build Cross-Session Evaluation Queries ---
# These queries specifically require information from PRIOR sessions to answer correctly.

@dataclass
class EvalQuery:
    query: str
    patient_id: str
    current_session_id: int
    answer_source_session: int  # Which session contains the answer
    expected_answer: str
    category: str  # 'allergy', 'medication', 'diagnosis', 'lab_result'
    is_safety_critical: bool

eval_queries = []

for patient in patients:
    pid = patient.patient_id

    # Allergy queries (safety-critical) -- test from session 3 looking back at session 1
    if patient.allergies:
        for allergy in patient.allergies:
            cross_reactive = ALLERGIES_DB[allergy]["cross_reactive"]
            if cross_reactive:
                drug = random.choice(cross_reactive)
                eval_queries.append(EvalQuery(
                    query=f"Can we prescribe {drug} for this patient?",
                    patient_id=pid,
                    current_session_id=3,
                    answer_source_session=1,
                    expected_answer=f"No -- patient has a documented allergy to {allergy}, "
                                    f"and {drug} is cross-reactive.",
                    category="allergy",
                    is_safety_critical=True,
                ))

    # Medication history queries -- test from session 3 looking back at session 1
    for med, dose in patient.medications.items():
        eval_queries.append(EvalQuery(
            query=f"What dose of {med} was the patient initially started on?",
            patient_id=pid,
            current_session_id=3,
            answer_source_session=1,
            expected_answer=f"The patient was initially started on {med} {dose}.",
            category="medication",
            is_safety_critical=False,
        ))

    # Lab trend queries -- test from session 4 looking back at session 1
    for test, val in patient.lab_results.items():
        eval_queries.append(EvalQuery(
            query=f"What was the patient's {test} at the initial visit?",
            patient_id=pid,
            current_session_id=4,
            answer_source_session=1,
            expected_answer=f"The patient's {test} was {val} {LAB_TESTS[test]['unit']} at the initial visit.",
            category="lab_result",
            is_safety_critical=False,
        ))

# Subsample to 30 queries for evaluation (balanced across categories)
random.shuffle(eval_queries)
eval_by_cat = defaultdict(list)
for q in eval_queries:
    eval_by_cat[q.category].append(q)

selected_queries = []
for cat, queries in eval_by_cat.items():
    # Take up to 8 from each category, prioritize safety-critical
    safety = [q for q in queries if q.is_safety_critical]
    non_safety = [q for q in queries if not q.is_safety_critical]
    selected_queries.extend(safety[:4])
    selected_queries.extend(non_safety[:max(0, 8 - len(safety[:4]))])

selected_queries = selected_queries[:30]
random.shuffle(selected_queries)

print(f"Total eval queries generated: {len(eval_queries)}")
print(f"Selected for evaluation: {len(selected_queries)}")
print(f"By category: {dict((cat, len([q for q in selected_queries if q.category == cat])) for cat in set(q.category for q in selected_queries))}")
print(f"Safety-critical: {sum(1 for q in selected_queries if q.is_safety_critical)}")
print(f"\nSample query:")
print(f"  Query: {selected_queries[0].query}")
print(f"  Patient: {selected_queries[0].patient_id}")
print(f"  Current session: {selected_queries[0].current_session_id}")
print(f"  Answer source: Session {selected_queries[0].answer_source_session}")
print(f"  Expected: {selected_queries[0].expected_answer}")

**Thought Questions:**
1. Why is it important that our evaluation queries specifically require cross-session memory? What would happen if we only tested within-session queries?
2. Why do we separate safety-critical queries (allergies) from informational queries (lab results)? How should evaluation weight them differently?
3. If a real clinical dataset were available, what additional entity types would you include beyond medications, allergies, diagnoses, and lab results?

---

## 3.2 Exploratory Data Analysis

In [ ]:
# --- Session Length Distribution ---
session_lengths = []
for s in all_sessions:
    total_tokens = sum(len(m.content.split()) for m in s.messages)
    session_lengths.append({
        "patient_id": s.patient_id,
        "session_id": s.session_id,
        "n_messages": len(s.messages),
        "total_tokens": total_tokens,
    })

df_sessions = pd.DataFrame(session_lengths)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Token distribution
axes[0].hist(df_sessions["total_tokens"], bins=20, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Tokens per Session")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Session Lengths")
axes[0].axvline(df_sessions["total_tokens"].median(), color="red", linestyle="--", label=f'Median: {df_sessions["total_tokens"].median():.0f}')
axes[0].legend()

# Messages per session
axes[1].hist(df_sessions["n_messages"], bins=10, color="coral", edgecolor="white")
axes[1].set_xlabel("Messages per Session")
axes[1].set_ylabel("Count")
axes[1].set_title("Messages per Session")

# Session length by session number
session_by_num = df_sessions.groupby("session_id")["total_tokens"].mean()
axes[2].bar(session_by_num.index, session_by_num.values, color="mediumseagreen", edgecolor="white")
axes[2].set_xlabel("Session Number")
axes[2].set_ylabel("Avg Tokens")
axes[2].set_title("Avg Session Length by Visit Number")

plt.tight_layout()
plt.show()

print(f"\nSession statistics:")
print(f"  Mean tokens/session: {df_sessions['total_tokens'].mean():.0f}")
print(f"  Median tokens/session: {df_sessions['total_tokens'].median():.0f}")
print(f"  95th percentile: {df_sessions['total_tokens'].quantile(0.95):.0f}")

In [ ]:
# --- Entity Type Distribution ---
all_entities_gt = []
for s in all_sessions:
    for e in s.entities_ground_truth:
        all_entities_gt.append({**e, "session_id": s.session_id, "patient_id": s.patient_id})

df_entities = pd.DataFrame(all_entities_gt)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Entity type counts
type_counts = df_entities["type"].value_counts()
axes[0].barh(type_counts.index, type_counts.values, color=["crimson", "steelblue", "mediumseagreen", "gold", "purple"])
axes[0].set_xlabel("Count")
axes[0].set_title("Entity Frequency by Type")

# Entity importance distribution
imp_counts = df_entities["importance"].value_counts().sort_index()
axes[1].bar(imp_counts.index.astype(str), imp_counts.values, color=["gray", "steelblue", "crimson"])
axes[1].set_xlabel("Importance Score")
axes[1].set_ylabel("Count")
axes[1].set_title("Entity Importance Distribution")

plt.tight_layout()
plt.show()

print(f"\nEntity statistics:")
print(f"  Total entities: {len(df_entities)}")
print(f"  Unique entity types: {df_entities['type'].nunique()}")
print(f"  Safety-critical (importance=1.0): {(df_entities['importance'] == 1.0).sum()}")
print(f"  Active clinical (importance=0.5): {(df_entities['importance'] == 0.5).sum()}")

In [ ]:
# --- Cross-Session Dependency Analysis ---
# What percentage of evaluation queries require information from prior sessions?

cross_session_rate = sum(
    1 for q in selected_queries
    if q.answer_source_session < q.current_session_id
) / len(selected_queries)

print(f"Cross-session dependency rate: {cross_session_rate:.1%}")
print(f"(This is the percentage of queries that REQUIRE prior-session memory)")

# Simulate what a sliding window of size K would miss
for K in [5, 10, 15, 20]:
    # Approximate: K turns = K messages. Session 1 has ~8 messages.
    # If we are in session 3, the window covers at most K messages back.
    # Session 1 info is ~16+ messages back from session 3.
    missed = 0
    for q in selected_queries:
        # Estimate messages between answer source and current session
        sessions_between = q.current_session_id - q.answer_source_session
        msgs_between = sessions_between * 8  # ~8 messages per session
        if msgs_between > K:
            missed += 1
    miss_rate = missed / len(selected_queries)
    print(f"  Window K={K:2d}: would miss {miss_rate:.1%} of cross-session facts")

**Thought Questions:**
1. Based on the cross-session dependency analysis, what is the minimum window size K that would capture all cross-session dependencies? Is this practical in terms of token cost?
2. Safety-critical entities (allergies) are rare compared to other entity types. Why does this make pure vector retrieval potentially dangerous for safety-critical recall?

---

## 3.3 Baseline Approach

Implement the sliding window baseline to establish a performance floor.

In [ ]:
# --- Sliding Window Baseline ---

class SlidingWindowBaseline:
    """Baseline memory system using only a sliding window + static patient summary.

    This is the simplest possible approach: keep the last K messages and a
    fixed patient summary. No long-term memory, no entity extraction,
    no semantic retrieval.
    """

    def __init__(self, window_size: int = 10):
        self.window_size = window_size
        self.messages = deque(maxlen=window_size * 2)
        self.patient_summary = ""

    def set_patient_summary(self, summary: str):
        """Set the static patient summary (pulled from EHR at session start)."""
        self.patient_summary = summary

    def add_message(self, role: str, content: str):
        """Add a message to the sliding window."""
        self.messages.append({"role": role, "content": content})

    def build_context(self, query: str) -> str:
        """Build baseline context: summary + window."""
        parts = []
        if self.patient_summary:
            parts.append(f"Patient Summary:\n{self.patient_summary}")
        if self.messages:
            window_text = "\n".join(
                f"{m['role']}: {m['content']}" for m in self.messages
            )
            parts.append(f"Recent Conversation:\n{window_text}")
        return "\n\n".join(parts)

    def get_token_count(self) -> int:
        """Estimate token count of current context."""
        context = self.build_context("")
        return len(context) // 4


def evaluate_baseline(patients, all_sessions, eval_queries, window_size=10):
    """Evaluate the sliding window baseline on cross-session queries.

    For each query:
    1. Replay all sessions for that patient up to current_session_id
    2. Build context using sliding window
    3. Check if the expected information is present in the context

    We use a simple string-matching heuristic: if key terms from the
    expected answer appear in the context, the memory "recalled" the fact.
    """
    results = []

    patient_lookup = {p.patient_id: p for p in patients}
    sessions_by_patient = defaultdict(list)
    for s in all_sessions:
        sessions_by_patient[s.patient_id].append(s)

    for q in eval_queries:
        baseline = SlidingWindowBaseline(window_size=window_size)

        # Set patient summary (minimal EHR snapshot)
        patient = patient_lookup[q.patient_id]
        summary = f"Patient: {patient.age}yo {patient.sex}. Diagnoses: {', '.join(patient.diagnoses[:2])}."
        baseline.set_patient_summary(summary)

        # Replay sessions
        for session in sessions_by_patient[q.patient_id]:
            if session.session_id > q.current_session_id:
                break
            for msg in session.messages:
                baseline.add_message(msg.role, msg.content)

        context = baseline.build_context(q.query)

        # Check if key information is in context
        # Extract key terms from expected answer
        expected_lower = q.expected_answer.lower()
        context_lower = context.lower()

        # For allergy queries, check if the allergen name is in context
        if q.category == "allergy":
            # Extract the allergen from the expected answer
            recalled = any(
                allergy in context_lower
                for allergy in ALLERGIES_DB.keys()
                if allergy in expected_lower
            )
        elif q.category == "medication":
            # Check if the specific dose is in context
            recalled = any(
                dose in context_lower
                for med_info in MEDICATIONS.values()
                for dose in med_info["common_doses"]
                if dose.lower() in expected_lower
            )
        else:
            # For lab results, check if the value is in context
            recalled = any(
                str(val) in context
                for val in patient.lab_results.values()
                if str(val) in q.expected_answer
            )

        results.append({
            "query": q.query,
            "patient_id": q.patient_id,
            "category": q.category,
            "is_safety_critical": q.is_safety_critical,
            "recalled": recalled,
            "context_tokens": len(context) // 4,
        })

    return pd.DataFrame(results)


# Evaluate baseline with different window sizes
baseline_results = {}
for K in [5, 10, 15, 20]:
    df_result = evaluate_baseline(patients, all_sessions, selected_queries, window_size=K)
    recall = df_result["recalled"].mean()
    safety_recall = df_result[df_result["is_safety_critical"]]["recalled"].mean() if df_result["is_safety_critical"].any() else 0
    avg_tokens = df_result["context_tokens"].mean()
    baseline_results[K] = {"recall": recall, "safety_recall": safety_recall, "avg_tokens": avg_tokens}
    print(f"Window K={K:2d}: Memory Recall={recall:.1%}, Safety Recall={safety_recall:.1%}, Avg Tokens={avg_tokens:.0f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ks = list(baseline_results.keys())
recalls = [baseline_results[k]["recall"] for k in ks]
safety_recalls = [baseline_results[k]["safety_recall"] for k in ks]
tokens = [baseline_results[k]["avg_tokens"] for k in ks]

axes[0].plot(ks, recalls, 'o-', color="steelblue", label="Overall Recall")
axes[0].plot(ks, safety_recalls, 's-', color="crimson", label="Safety Recall")
axes[0].set_xlabel("Window Size K")
axes[0].set_ylabel("Recall")
axes[0].set_title("Baseline Memory Recall vs Window Size")
axes[0].legend()
axes[0].set_ylim(0, 1.05)

axes[1].bar(ks, tokens, color="mediumseagreen", edgecolor="white")
axes[1].set_xlabel("Window Size K")
axes[1].set_ylabel("Avg Context Tokens")
axes[1].set_title("Token Cost vs Window Size")

plt.tight_layout()
plt.show()

**Thought Questions:**
1. At what window size does the baseline first achieve reasonable recall? Is it practical given context window limits?
2. Why is the safety recall consistently low regardless of window size for cross-session allergy queries?
3. What fundamental limitation makes sliding window memory unsuitable for clinical applications?

---

## 3.4 Model Design: Hybrid Memory Pipeline

This is the core implementation section. You will build all four memory subsystems and the orchestration layer.

### Subsystem 1: Sliding Window Memory

In [ ]:
# --- TODO 1: Sliding Window Memory ---

class SlidingWindowMemory:
    """Maintains the last K conversation turns for immediate context.

    Args:
        window_size: Number of turn pairs (user + assistant) to retain

    Hints:
        1. Use a deque with maxlen = window_size * 2 (user + assistant messages)
        2. add_message appends a role/content dict to the deque
        3. get_context joins all messages with role prefixes
        4. get_token_count estimates tokens using len(text) // 4
    """

    def __init__(self, window_size: int = 10):
        # TODO: Implement
        pass

    def add_message(self, role: str, content: str):
        """Add a message to the sliding window.

        Args:
            role: 'physician' or 'assistant'
            content: The message text
        """
        # TODO: Implement
        pass

    def get_context(self) -> str:
        """Return all messages in the window as a formatted string.

        Format each message as: "ROLE: content"
        Join with newlines.
        """
        # TODO: Implement
        pass

    def get_token_count(self) -> int:
        """Estimate the token count of the current window contents."""
        # TODO: Implement
        pass

    def clear(self):
        """Clear the window (call at session boundaries if needed)."""
        # TODO: Implement
        pass


# --- Verification ---
sw = SlidingWindowMemory(window_size=3)
sw.add_message("physician", "What medications is the patient on?")
sw.add_message("assistant", "The patient is currently on metformin 500mg twice daily.")
sw.add_message("physician", "Any allergies?")
sw.add_message("assistant", "The patient has a documented allergy to penicillin.")
sw.add_message("physician", "What about sulfa drugs?")
sw.add_message("assistant", "No documented sulfa drug allergies.")

context = sw.get_context()
assert context is not None, "get_context returned None -- did you implement it?"
assert "metformin" in context, "Expected metformin in context"
assert "penicillin" in context, "Expected penicillin in context"
token_count = sw.get_token_count()
assert token_count > 0, "Token count should be positive"

# Test overflow: add 4th pair, should drop the oldest pair
sw.add_message("physician", "Start lisinopril 10mg daily.")
sw.add_message("assistant", "Lisinopril 10mg daily initiated.")
context_after = sw.get_context()
# With window_size=3, oldest pair should be dropped
# The first pair about "what medications" might be dropped

print(f"Sliding window verified: {token_count} tokens")
print(f"Context preview:\n{context[:300]}...")

### Subsystem 2: Vector-Backed Long-Term Memory

In [ ]:
# --- TODO 2: Vector-Backed Long-Term Memory ---

import faiss

# Load a lightweight embedding model (fits on Colab free tier)
from sentence_transformers import SentenceTransformer
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
EMBED_DIM = 384  # Dimension of all-MiniLM-L6-v2

def embed_text(text: str) -> np.ndarray:
    """Embed text using the sentence transformer model."""
    return embed_model.encode(text, normalize_embeddings=True)


class VectorMemory:
    """Stores all conversation messages as embeddings for semantic retrieval.

    Args:
        embedding_fn: Function that maps text -> numpy array of shape (dim,)
        dim: Embedding dimension (default 384 for all-MiniLM-L6-v2)
        top_k: Default number of results to retrieve

    Hints:
        1. Use faiss.IndexFlatIP for inner product (cosine sim on normalized vectors)
        2. Normalize vectors before adding: vec / np.linalg.norm(vec)
           (embed_text already normalizes, but be safe)
        3. Store messages + metadata in a parallel list (same index as FAISS)
        4. In retrieve(): encode the query, search the index, return results
        5. Filter out FAISS results with index -1 (unfilled slots when
           store has fewer items than top_k)
    """

    def __init__(self, embedding_fn, dim: int = 384, top_k: int = 5):
        # TODO: Implement
        # Create self.index = faiss.IndexFlatIP(dim)
        # Create self.messages = []  for parallel storage
        pass

    def add_message(self, role: str, content: str, metadata: dict = None):
        """Embed and store a message with metadata.

        Args:
            role: 'physician' or 'assistant'
            content: The message text
            metadata: Dict with keys like 'session_id', 'timestamp', 'importance'

        Hints:
            1. Embed the content using self.embedding_fn
            2. Reshape to (1, dim) and ensure float32 dtype
            3. Add to FAISS index
            4. Append message dict to self.messages list
        """
        # TODO: Implement
        pass

    def retrieve(self, query: str, top_k: int = None) -> List[Dict]:
        """Retrieve the top-k most similar messages to the query.

        Args:
            query: The physician's current question
            top_k: Override default if provided

        Returns:
            List of dicts: {'role', 'content', 'score', 'metadata'}
            Sorted by similarity score descending.

        Hints:
            1. Embed the query
            2. Use self.index.search(query_vec, k)
            3. Iterate over returned indices and scores
            4. Skip indices that are -1 (no result)
            5. Build result dicts from self.messages[idx]
        """
        # TODO: Implement
        pass

    def get_store_size(self) -> int:
        """Return the number of messages currently stored."""
        # TODO: Implement
        pass


# --- Verification ---
vm = VectorMemory(embedding_fn=embed_text, dim=EMBED_DIM, top_k=3)

# Add messages from different sessions
vm.add_message("physician", "Patient has a severe allergy to penicillin.",
               metadata={"session_id": 1, "timestamp": "2024-01-15T09:00:00", "importance": 1.0})
vm.add_message("assistant", "Penicillin allergy documented. Flagged as safety-critical.",
               metadata={"session_id": 1, "timestamp": "2024-01-15T09:01:00", "importance": 1.0})
vm.add_message("physician", "Start metformin 500mg twice daily for diabetes.",
               metadata={"session_id": 1, "timestamp": "2024-01-15T09:05:00", "importance": 0.5})
vm.add_message("assistant", "Metformin 500mg BID initiated.",
               metadata={"session_id": 1, "timestamp": "2024-01-15T09:06:00", "importance": 0.5})
vm.add_message("physician", "How is the patient responding to the exercise program?",
               metadata={"session_id": 2, "timestamp": "2024-02-01T09:00:00", "importance": 0.0})
vm.add_message("physician", "A1C came back at 7.2 percent.",
               metadata={"session_id": 2, "timestamp": "2024-02-01T09:05:00", "importance": 0.5})

assert vm.get_store_size() == 6, f"Expected 6 messages, got {vm.get_store_size()}"

# Test retrieval: allergy query should find penicillin messages
results = vm.retrieve("Does this patient have any drug allergies?")
assert results is not None, "retrieve() returned None"
assert len(results) > 0, "retrieve() returned empty list"
assert any("penicillin" in r["content"].lower() for r in results), \
    "Penicillin allergy should be the top result for an allergy query"

# Test retrieval: medication query
results2 = vm.retrieve("What diabetes medications is the patient taking?")
assert any("metformin" in r["content"].lower() for r in results2), \
    "Metformin should be retrieved for a diabetes medication query"

print(f"Vector store verified: {vm.get_store_size()} messages")
print(f"\nAllergy query results:")
for r in results:
    print(f"  [{r['score']:.3f}] {r['role']}: {r['content'][:80]}...")
print(f"\nMedication query results:")
for r in results2:
    print(f"  [{r['score']:.3f}] {r['role']}: {r['content'][:80]}...")

### Subsystem 3: Entity Store

In [ ]:
# --- TODO 3: Entity Store ---

@dataclass
class ClinicalEntity:
    name: str
    entity_type: str  # 'allergy', 'medication', 'diagnosis', 'lab_result', 'preference'
    value: str
    session_id: int
    timestamp: str
    importance: float  # 1.0 = safety-critical, 0.5 = active clinical, 0.0 = informational
    is_active: bool = True
    notes: str = ""


class EntityStore:
    """Structured store for clinical entities extracted from conversations.

    This is the MOST IMPORTANT subsystem for patient safety. It guarantees
    that allergy information is never lost, regardless of how many sessions
    have passed or how the conversation has evolved.

    Hints:
        1. Use a dict keyed by (entity_type, normalized_name) for fast lookup
        2. When updating an existing entity, keep the old value in notes
        3. Default importance: allergy=1.0, medication=0.5, diagnosis=0.5,
           lab_result=0.5, preference=0.0
        4. extract_entities uses regex patterns -- start simple, iterate
        5. get_safety_critical returns ALL entities with importance >= 1.0
        6. format_for_context produces organized, compact text
    """

    IMPORTANCE_DEFAULTS = {
        "allergy": 1.0,
        "medication": 0.5,
        "diagnosis": 0.5,
        "lab_result": 0.5,
        "preference": 0.0,
    }

    def __init__(self):
        # TODO: Implement
        # self.entities = {}  # key: (entity_type, name_normalized) -> ClinicalEntity
        pass

    def add_entity(self, entity: ClinicalEntity):
        """Add or update a clinical entity.

        If entity with same (type, name) exists:
        - Update the value
        - Append old value to notes with timestamp
        - Update session_id and timestamp to the new values

        Args:
            entity: The ClinicalEntity to add or update
        """
        # TODO: Implement
        pass

    def extract_entities(self, text: str, session_id: int, timestamp: str) -> List[ClinicalEntity]:
        """Extract clinical entities from conversation text using regex.

        Patterns to detect:
        - Allergies: "allerg(y|ic) to X", "allergy: X", "X allergy"
        - Medications: "start(ed)? X dose", "taking X", "prescribe(d) X",
                       "X dose initiated", "change X from dose1 to dose2"
        - Diagnoses: "diagnos(ed|is) (with)? X", "diagnosed: X"
        - Lab results: "X (is|was|came back|:) value unit"

        Args:
            text: The conversation text
            session_id: Session number
            timestamp: ISO timestamp

        Returns:
            List of ClinicalEntity objects (may be empty, never None)

        Hints:
            1. Normalize text to lowercase for matching
            2. For medications, check against MEDICATIONS dict keys
            3. For allergies, check against ALLERGIES_DB dict keys
            4. Use re.IGNORECASE flag
            5. Assign importance from IMPORTANCE_DEFAULTS
        """
        # TODO: Implement
        pass

    def get_by_type(self, entity_type: str) -> List[ClinicalEntity]:
        """Return all active entities of a given type.

        Args:
            entity_type: One of 'allergy', 'medication', 'diagnosis', 'lab_result', 'preference'
        """
        # TODO: Implement
        pass

    def get_safety_critical(self) -> List[ClinicalEntity]:
        """Return all entities with importance >= 1.0.

        These MUST be included in every context, regardless of query relevance.
        """
        # TODO: Implement
        pass

    def format_for_context(self, max_tokens: int = 500) -> str:
        """Format all active entities as a compact string for context injection.

        Output format:
        ## SAFETY ALERTS (always included)
        - ALLERGY: Penicillin -- severe reaction (Session 1, 2024-01-15)

        ## Active Medications
        - Metformin 500mg BID (Session 1, 2024-01-15)
        - Lisinopril 10mg daily (Session 2, 2024-02-01)

        ## Active Diagnoses
        - Type 2 Diabetes Mellitus (Session 1)

        ## Recent Lab Results
        - A1C: 7.2% (Session 2, 2024-02-01)

        Hints:
            1. Safety-critical entities go first, in a clearly marked section
            2. Group remaining entities by type
            3. Include session_id and timestamp for traceability
            4. Truncate if exceeding max_tokens
        """
        # TODO: Implement
        pass

    def get_all_entities(self) -> List[ClinicalEntity]:
        """Return all entities (active and inactive)."""
        # TODO: Implement
        pass


# --- Verification ---
es = EntityStore()

# Test extraction
entities = es.extract_entities(
    "Patient has a severe allergy to penicillin. "
    "Started metformin 500mg twice daily for diabetes management. "
    "Diagnosed with Type 2 Diabetes. "
    "A1C came back at 7.2 percent.",
    session_id=1,
    timestamp="2024-01-15T10:00:00"
)
assert entities is not None, "extract_entities returned None"
assert len(entities) >= 3, f"Expected >= 3 entities, got {len(entities)}"

for e in entities:
    es.add_entity(e)

# Test safety-critical
safety = es.get_safety_critical()
assert len(safety) >= 1, "Must find penicillin allergy as safety-critical"
assert any("penicillin" in e.name.lower() for e in safety), "Penicillin must be in safety entities"

# Test medication retrieval
meds = es.get_by_type("medication")
assert len(meds) >= 1, "Must find at least 1 medication"

# Test context formatting
ctx = es.format_for_context()
assert ctx is not None and len(ctx) > 0, "Context should not be empty"
assert "SAFETY" in ctx.upper() or "ALLERGY" in ctx.upper(), "Safety section must be present"

print(f"Extracted {len(entities)} entities:")
for e in entities:
    print(f"  [{e.entity_type}] {e.name}: {e.value} (importance={e.importance})")
print(f"\nSafety-critical: {len(safety)}")
print(f"\nFormatted context:\n{ctx}")

### Subsystem 4: Running Summary

In [ ]:
# --- TODO 4: Running Summary ---

class RunningSummary:
    """Maintains a running extractive summary of the patient's clinical trajectory.

    After each session, the summary is updated by selecting the most important
    sentences from the combined old summary + new session text, scored by
    clinical relevance.

    Args:
        max_summary_tokens: Maximum token budget for the summary

    Hints:
        1. Store current summary as a string
        2. For update: split old summary + new text into sentences
        3. Score each sentence:
           - +3.0 if contains safety keywords (allergy, reaction, contraindicated)
           - +2.0 if contains medication keywords (started, prescribed, mg, dose)
           - +1.5 if contains diagnosis keywords (diagnosed, diagnosis, condition)
           - +1.0 if contains lab keywords (A1C, LDL, TSH, creatinine, result)
           - +0.5 for all other sentences
        4. Select top sentences by score until max_tokens budget is reached
        5. Maintain chronological order in the final summary
    """

    SAFETY_KEYWORDS = ["allergy", "allergic", "reaction", "anaphylaxis",
                       "contraindicated", "adverse", "warning", "severe"]
    MED_KEYWORDS = ["started", "prescribed", "mg", "dose", "medication",
                    "adjusted", "changed", "discontinued", "initiated"]
    DIAG_KEYWORDS = ["diagnosed", "diagnosis", "condition", "disease",
                     "syndrome", "cancer", "positive", "negative"]
    LAB_KEYWORDS = ["a1c", "ldl", "tsh", "creatinine", "egfr", "inr",
                    "bnp", "result", "lab", "percent"]

    def __init__(self, max_summary_tokens: int = 500):
        # TODO: Implement
        pass

    def _score_sentence(self, sentence: str) -> float:
        """Score a sentence by clinical importance.

        Args:
            sentence: A single sentence from session text

        Returns:
            Float score (higher = more important)
        """
        # TODO: Implement
        pass

    def update_summary(self, session_text: str, session_id: int):
        """Update the running summary with content from a new session.

        Args:
            session_text: Full text of the completed session
            session_id: Session number for tracking

        Hints:
            1. Combine current summary + new session text
            2. Split into sentences (split on '. ' or use regex)
            3. Score each sentence
            4. Select top sentences until token budget is reached
            5. Preserve chronological order
            6. Prefix with "Session N:" markers for traceability
        """
        # TODO: Implement
        pass

    def get_summary(self) -> str:
        """Return the current summary text."""
        # TODO: Implement
        pass

    def get_token_count(self) -> int:
        """Estimate token count of current summary."""
        # TODO: Implement
        pass


# --- Verification ---
rs = RunningSummary(max_summary_tokens=300)

rs.update_summary(
    "Session started. Patient reports feeling well. "
    "Patient has a severe allergy to penicillin -- documented and flagged. "
    "Started metformin 500mg twice daily for Type 2 Diabetes. "
    "A1C was 7.2 percent, target is below 7.0 percent. "
    "Follow-up scheduled in 4 weeks.",
    session_id=1
)

summary = rs.get_summary()
assert summary is not None and len(summary) > 0, "Summary should not be empty after update"
assert "allergy" in summary.lower() or "penicillin" in summary.lower(), \
    "Summary MUST preserve allergy information (safety-critical)"

rs.update_summary(
    "Patient reports GI upset with metformin. "
    "Adjusted metformin from 500mg to 250mg due to side effects. "
    "A1C improved to 6.9 percent. Patient feeling better overall.",
    session_id=2
)

summary2 = rs.get_summary()
assert "allergy" in summary2.lower() or "penicillin" in summary2.lower(), \
    "Allergy info must persist after second session update"
assert rs.get_token_count() <= 300, f"Summary exceeds budget: {rs.get_token_count()} > 300"

print(f"Running summary ({rs.get_token_count()} tokens):")
print(summary2)

### Subsystem 5: Hybrid Memory Orchestrator

In [ ]:
# --- TODO 5: Hybrid Memory Orchestrator ---

class HybridMemorySystem:
    """Orchestrates four memory subsystems into a unified clinical memory.

    When a physician asks a question:
    1. Safety-critical entities from EntityStore are ALWAYS included (non-negotiable)
    2. Running summary provides big-picture trajectory
    3. Vector store retrieves semantically relevant past conversations
    4. Sliding window provides immediate conversational context
    5. All memories are scored using the hybrid scoring function
    6. Context is assembled with XML-tagged sections and token budgeting

    Args:
        embedding_fn: Function mapping text -> numpy array
        window_size: Sliding window size (turn pairs)
        vector_top_k: Number of vector retrieval results
        max_context_tokens: Total token budget for assembled context

    Hints:
        1. Initialize all four subsystems in __init__
        2. add_message propagates to sliding window, vector store, and entity store
        3. end_session updates the running summary
        4. compute_hybrid_score implements:
           S = 0.45 * similarity + 0.20 * exp(-0.01 * recency_days) + 0.35 * importance
        5. build_context:
           a. Start with safety entities (ALWAYS, first, non-negotiable)
           b. Add running summary
           c. Score and rank vector results using hybrid score
           d. Add top vector results
           e. Add sliding window
           f. Wrap each section in XML tags
           g. Enforce token budget
    """

    def __init__(self, embedding_fn, window_size: int = 10,
                 vector_top_k: int = 5, max_context_tokens: int = 8000):
        # TODO: Implement
        # Initialize: self.window, self.vector_store, self.entity_store, self.summary
        pass

    def add_message(self, role: str, content: str, session_id: int, timestamp: str):
        """Add a message to all relevant subsystems.

        This should:
        1. Add to sliding window
        2. Add to vector store (with metadata)
        3. Extract entities and add to entity store
        """
        # TODO: Implement
        pass

    def end_session(self, session_id: int, session_text: str):
        """Called when a session ends. Updates the running summary.

        Also clears the sliding window for the next session.
        """
        # TODO: Implement
        pass

    def compute_hybrid_score(self, similarity: float, recency_days: float,
                              importance: float) -> float:
        """Compute hybrid memory score.

        Formula: S = 0.45 * similarity + 0.20 * exp(-0.01 * recency_days) + 0.35 * importance

        Args:
            similarity: Cosine similarity [0, 1]
            recency_days: Days since memory creation
            importance: Clinical importance [0.0, 0.5, 1.0]

        Returns:
            Combined score (higher = more relevant)
        """
        # TODO: Implement
        pass

    def build_context(self, query: str, current_session_id: int) -> str:
        """Build the complete context for a physician query.

        Structure:
        <safety_alerts>
          [All safety-critical entities -- ALWAYS included]
        </safety_alerts>

        <patient_summary>
          [Running summary of clinical trajectory]
        </patient_summary>

        <relevant_history>
          [Top vector retrieval results, scored and ranked]
        </relevant_history>

        <recent_conversation>
          [Sliding window contents]
        </recent_conversation>

        Token budget allocation:
        - Safety alerts: unlimited (always included, typically < 200 tokens)
        - Summary: up to 500 tokens
        - Retrieved history: up to 40% of remaining budget
        - Recent conversation: remaining budget

        Hints:
            1. Start with safety entities (get_safety_critical + format)
            2. Add summary (get_summary)
            3. Retrieve from vector store, compute hybrid scores, sort
            4. Add top-scored memories until budget is reached
            5. Add sliding window context
            6. Wrap each section in XML tags
            7. Verify total does not exceed max_context_tokens
        """
        # TODO: Implement
        pass

    def get_memory_stats(self) -> Dict:
        """Return statistics about the current memory state."""
        # TODO: Implement
        pass


# --- Verification: Multi-Session Test ---
hm = HybridMemorySystem(
    embedding_fn=embed_text,
    window_size=5,
    vector_top_k=5,
    max_context_tokens=4000
)

# Session 1: Initial visit -- establish baseline
session1_msgs = [
    ("physician", "Let's start with allergies. Any known drug allergies?", "2024-01-15T09:00:00"),
    ("assistant", "Patient reports severe allergy to penicillin. Documented as safety-critical.", "2024-01-15T09:01:00"),
    ("physician", "Current medications?", "2024-01-15T09:05:00"),
    ("assistant", "Patient is on metformin 500mg twice daily and lisinopril 10mg daily.", "2024-01-15T09:06:00"),
    ("physician", "Latest A1C?", "2024-01-15T09:10:00"),
    ("assistant", "A1C is 7.2 percent. Target is below 7.0.", "2024-01-15T09:11:00"),
]

session1_text_parts = []
for role, content, ts in session1_msgs:
    hm.add_message(role, content, session_id=1, timestamp=ts)
    session1_text_parts.append(f"{role}: {content}")
hm.end_session(1, " ".join(session1_text_parts))

# Session 2: Follow-up -- medication adjustment
session2_msgs = [
    ("physician", "Patient reports GI upset with metformin. Reduce to 250mg.", "2024-02-01T09:00:00"),
    ("assistant", "Metformin reduced from 500mg to 250mg BID due to GI side effects.", "2024-02-01T09:01:00"),
    ("physician", "New A1C results?", "2024-02-01T09:05:00"),
    ("assistant", "A1C improved to 6.9 percent. Good progress toward target.", "2024-02-01T09:06:00"),
]

session2_text_parts = []
for role, content, ts in session2_msgs:
    hm.add_message(role, content, session_id=2, timestamp=ts)
    session2_text_parts.append(f"{role}: {content}")
hm.end_session(2, " ".join(session2_text_parts))

# Session 3: Test cross-session memory
# Test 1: Allergy recall (safety-critical)
context_allergy = hm.build_context(
    "Can we prescribe amoxicillin for the patient's infection?",
    current_session_id=3
)
assert context_allergy is not None, "build_context returned None"
assert "penicillin" in context_allergy.lower() or "allergy" in context_allergy.lower(), \
    "CRITICAL FAILURE: Penicillin allergy must appear in context when amoxicillin is discussed"
assert "safety" in context_allergy.lower() or "alert" in context_allergy.lower(), \
    "Safety alerts section must be present"

# Test 2: Medication history recall
context_med = hm.build_context(
    "What was the original metformin dosage before the adjustment?",
    current_session_id=3
)
assert "500" in context_med, \
    "Original metformin dose (500mg) from Session 1 should be retrievable"

# Test 3: Lab trend recall
context_lab = hm.build_context(
    "What has the A1C trend been?",
    current_session_id=3
)
assert "7.2" in context_lab or "6.9" in context_lab, \
    "A1C values should be retrievable for trend queries"

print("All multi-session memory tests PASSED!")
print(f"\nMemory stats: {hm.get_memory_stats()}")
print(f"\n--- Allergy Query Context (first 600 chars) ---")
print(context_allergy[:600])

---

## 3.5 Training Strategy

In this section, you will fine-tune the embedding model on clinical query-passage pairs to improve retrieval quality for medical text.

In [ ]:
# --- Contrastive Fine-tuning for Clinical Embeddings ---
# We create positive pairs (query, relevant passage) and hard negatives
# (query, similar-but-incorrect passage) from our synthetic sessions.

def create_training_pairs(patients, all_sessions):
    """Create contrastive training pairs from the session data.

    Positive pairs: (physician question, assistant answer from same exchange)
    Hard negatives: (physician question, assistant answer from DIFFERENT patient
                     with similar medical content)
    """
    positives = []
    all_assistant_msgs = []

    for session in all_sessions:
        msgs = session.messages
        for i in range(0, len(msgs) - 1, 2):
            if msgs[i].role == "physician" and i+1 < len(msgs) and msgs[i+1].role == "assistant":
                positives.append((msgs[i].content, msgs[i+1].content, session.patient_id))
                all_assistant_msgs.append((msgs[i+1].content, session.patient_id))

    # Create hard negatives: similar medical content from different patients
    training_data = []
    for query, positive, pid in positives:
        # Find hard negatives: assistant messages from other patients
        negatives = [msg for msg, other_pid in all_assistant_msgs if other_pid != pid]
        if len(negatives) >= 3:
            hard_negs = random.sample(negatives, 3)
            training_data.append({
                "query": query,
                "positive": positive,
                "negatives": hard_negs,
            })

    return training_data

training_pairs = create_training_pairs(patients, all_sessions)
print(f"Created {len(training_pairs)} training pairs")
print(f"\nSample pair:")
print(f"  Query: {training_pairs[0]['query']}")
print(f"  Positive: {training_pairs[0]['positive'][:80]}...")
print(f"  Negative 1: {training_pairs[0]['negatives'][0][:80]}...")

In [ ]:
# --- TODO 6: InfoNCE Contrastive Loss ---

def info_nce_loss(query_emb: np.ndarray, positive_emb: np.ndarray,
                  negative_embs: np.ndarray, temperature: float = 0.07) -> float:
    """Compute the InfoNCE contrastive loss.

    L = -log(exp(sim(q, p+) / tau) / (exp(sim(q, p+) / tau) + sum(exp(sim(q, n-) / tau))))

    Args:
        query_emb: Query embedding, shape (dim,)
        positive_emb: Positive passage embedding, shape (dim,)
        negative_embs: Negative passage embeddings, shape (n_neg, dim)
        temperature: Temperature parameter (default 0.07)

    Returns:
        Scalar loss value

    Hints:
        1. Compute cosine similarity between query and positive
        2. Compute cosine similarities between query and each negative
        3. Apply temperature scaling: sim / tau
        4. Use log-sum-exp trick for numerical stability:
           log(sum(exp(x))) = max(x) + log(sum(exp(x - max(x))))
        5. Return the negative log probability of the positive
    """
    # TODO: Implement
    pass


# --- Verification ---
q = np.random.randn(384).astype(np.float32)
q = q / np.linalg.norm(q)
p = q + np.random.randn(384).astype(np.float32) * 0.1
p = p / np.linalg.norm(p)
negs = np.random.randn(3, 384).astype(np.float32)
negs = negs / np.linalg.norm(negs, axis=1, keepdims=True)

loss = info_nce_loss(q, p, negs)
assert loss is not None, "info_nce_loss returned None"
assert isinstance(loss, float), f"Expected float, got {type(loss)}"
assert loss > 0, "InfoNCE loss should be positive"
assert loss < 10, f"Loss seems too high: {loss}"
print(f"InfoNCE loss: {loss:.4f}")

# Loss should be lower when positive is very similar to query
q2 = q.copy()
p2 = q2 + np.random.randn(384).astype(np.float32) * 0.001  # Very similar
p2 = p2 / np.linalg.norm(p2)
loss2 = info_nce_loss(q2, p2, negs)
assert loss2 < loss, "Loss should decrease when positive is more similar to query"
print(f"InfoNCE loss (very similar positive): {loss2:.4f}")
print("InfoNCE loss verified!")

In [ ]:
# --- Training Loop (Lightweight, Colab-compatible) ---
# We compute embeddings and losses but do NOT actually fine-tune the model
# (fine-tuning a transformer requires GPU memory beyond Colab free tier).
# Instead, we demonstrate the training loop structure and loss computation.

losses = []
for epoch in range(3):
    epoch_losses = []
    random.shuffle(training_pairs)

    for pair in training_pairs[:50]:  # Subsample for speed
        q_emb = embed_text(pair["query"])
        p_emb = embed_text(pair["positive"])
        n_embs = np.array([embed_text(n) for n in pair["negatives"]])

        loss = info_nce_loss(q_emb, p_emb, n_embs)
        epoch_losses.append(loss)

    mean_loss = np.mean(epoch_losses)
    losses.append(mean_loss)
    print(f"Epoch {epoch+1}: Mean InfoNCE Loss = {mean_loss:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(range(1, 4), losses, 'o-', color="steelblue", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Mean InfoNCE Loss")
plt.title("Contrastive Training Loss (Computed, Not Backpropagated)")
plt.grid(True, alpha=0.3)
plt.show()

---

## 3.6 Evaluation

In [ ]:
# --- Full Evaluation: Hybrid Memory vs Baseline ---

def evaluate_hybrid(patients, all_sessions, eval_queries, embedding_fn):
    """Evaluate the hybrid memory system on cross-session queries.

    For each patient:
    1. Replay all sessions into the hybrid memory system
    2. For each query targeting that patient, build context and check recall
    """
    results = []
    sessions_by_patient = defaultdict(list)
    for s in all_sessions:
        sessions_by_patient[s.patient_id].append(s)

    patient_lookup = {p.patient_id: p for p in patients}
    patient_memories = {}

    # Build memory for each patient
    for pid in tqdm(set(q.patient_id for q in eval_queries), desc="Building patient memories"):
        hm = HybridMemorySystem(
            embedding_fn=embedding_fn,
            window_size=5,
            vector_top_k=5,
            max_context_tokens=4000
        )

        for session in sessions_by_patient[pid]:
            text_parts = []
            for msg in session.messages:
                hm.add_message(msg.role, msg.content, session.session_id, msg.timestamp)
                text_parts.append(f"{msg.role}: {msg.content}")
            hm.end_session(session.session_id, " ".join(text_parts))

        patient_memories[pid] = hm

    # Evaluate queries
    for q in eval_queries:
        hm = patient_memories.get(q.patient_id)
        if hm is None:
            continue

        context = hm.build_context(q.query, q.current_session_id)
        context_lower = context.lower()
        expected_lower = q.expected_answer.lower()

        # Check recall based on category
        if q.category == "allergy":
            recalled = any(
                allergy in context_lower
                for allergy in ALLERGIES_DB.keys()
                if allergy in expected_lower
            )
        elif q.category == "medication":
            recalled = any(
                dose in context_lower
                for med_info in MEDICATIONS.values()
                for dose in med_info["common_doses"]
                if dose.lower() in expected_lower
            )
        else:
            recalled = any(
                str(val) in context
                for p in patients if p.patient_id == q.patient_id
                for val in p.lab_results.values()
                if str(val) in q.expected_answer
            )

        results.append({
            "query": q.query,
            "patient_id": q.patient_id,
            "category": q.category,
            "is_safety_critical": q.is_safety_critical,
            "recalled": recalled,
            "context_tokens": len(context) // 4,
        })

    return pd.DataFrame(results)


# Run evaluation
df_hybrid = evaluate_hybrid(patients, all_sessions, selected_queries, embed_text)

# Compare with baseline (K=10)
df_baseline = evaluate_baseline(patients, all_sessions, selected_queries, window_size=10)

# Summary table
print("=" * 70)
print("EVALUATION RESULTS: Hybrid Memory vs Sliding Window Baseline")
print("=" * 70)

metrics = {}
for name, df in [("Baseline (K=10)", df_baseline), ("Hybrid Memory", df_hybrid)]:
    overall_recall = df["recalled"].mean()
    safety_recall = df[df["is_safety_critical"]]["recalled"].mean() if df["is_safety_critical"].any() else 0
    med_recall = df[df["category"] == "medication"]["recalled"].mean() if (df["category"] == "medication").any() else 0
    lab_recall = df[df["category"] == "lab_result"]["recalled"].mean() if (df["category"] == "lab_result").any() else 0
    avg_tokens = df["context_tokens"].mean()

    metrics[name] = {
        "Overall Recall": overall_recall,
        "Safety Recall": safety_recall,
        "Medication Recall": med_recall,
        "Lab Result Recall": lab_recall,
        "Avg Context Tokens": avg_tokens,
    }

df_comparison = pd.DataFrame(metrics).T
print(df_comparison.to_string(float_format="{:.1%}".format))

In [ ]:
# --- Visualization ---

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart comparison
metric_names = ["Overall Recall", "Safety Recall", "Medication Recall", "Lab Result Recall"]
x = np.arange(len(metric_names))
width = 0.35

baseline_vals = [metrics["Baseline (K=10)"][m] for m in metric_names]
hybrid_vals = [metrics["Hybrid Memory"][m] for m in metric_names]

axes[0].bar(x - width/2, baseline_vals, width, label="Baseline (K=10)", color="lightcoral")
axes[0].bar(x + width/2, hybrid_vals, width, label="Hybrid Memory", color="steelblue")
axes[0].set_ylabel("Recall")
axes[0].set_title("Memory Recall: Baseline vs Hybrid")
axes[0].set_xticks(x)
axes[0].set_xticklabels(metric_names, rotation=15, ha="right")
axes[0].legend()
axes[0].set_ylim(0, 1.1)
axes[0].axhline(y=0.92, color="green", linestyle="--", alpha=0.5, label="Target (92%)")

# Token efficiency
token_data = {
    "Baseline": metrics["Baseline (K=10)"]["Avg Context Tokens"],
    "Hybrid": metrics["Hybrid Memory"]["Avg Context Tokens"],
}
recall_data = {
    "Baseline": metrics["Baseline (K=10)"]["Overall Recall"],
    "Hybrid": metrics["Hybrid Memory"]["Overall Recall"],
}
axes[1].scatter(token_data["Baseline"], recall_data["Baseline"], s=200, color="lightcoral",
                label="Baseline", zorder=5, edgecolors="black")
axes[1].scatter(token_data["Hybrid"], recall_data["Hybrid"], s=200, color="steelblue",
                label="Hybrid", zorder=5, edgecolors="black")
axes[1].set_xlabel("Avg Context Tokens")
axes[1].set_ylabel("Overall Recall")
axes[1].set_title("Token Efficiency: Recall per Token")
axes[1].legend()

plt.tight_layout()
plt.show()

---

## 3.7 Error Analysis

In [ ]:
# --- Error Categorization ---

def categorize_errors(df_results, name="System"):
    """Categorize errors into clinical failure types."""
    errors = df_results[~df_results["recalled"]].copy()

    if len(errors) == 0:
        print(f"\n{name}: No errors detected!")
        return pd.DataFrame()

    error_categories = []
    for _, row in errors.iterrows():
        if row["is_safety_critical"]:
            category = "AMNESIA (Safety-Critical)"
            severity = 3  # Patient safety hazard
        elif row["category"] == "medication":
            category = "AMNESIA (Medication History)"
            severity = 2  # Clinical risk
        elif row["category"] == "lab_result":
            category = "STALENESS (Lab Value)"
            severity = 1  # Inconvenience
        else:
            category = "AMNESIA (General)"
            severity = 1

        error_categories.append({
            "query": row["query"][:60] + "...",
            "category": row["category"],
            "error_type": category,
            "severity": severity,
            "patient_id": row["patient_id"],
        })

    df_errors = pd.DataFrame(error_categories)

    print(f"\n{'='*60}")
    print(f"ERROR ANALYSIS: {name}")
    print(f"{'='*60}")
    print(f"Total queries: {len(df_results)}")
    print(f"Total errors: {len(df_errors)}")
    print(f"Error rate: {len(df_errors)/len(df_results):.1%}")

    print(f"\nErrors by type:")
    for etype, group in df_errors.groupby("error_type"):
        avg_sev = group["severity"].mean()
        print(f"  {etype}: {len(group)} errors (avg severity: {avg_sev:.1f}/3)")

    print(f"\nErrors by category:")
    for cat, group in df_errors.groupby("category"):
        print(f"  {cat}: {len(group)} errors")

    return df_errors


df_errors_baseline = categorize_errors(df_baseline, "Baseline (K=10)")
df_errors_hybrid = categorize_errors(df_hybrid, "Hybrid Memory")

In [ ]:
# --- Thought exercise: Propose fixes for top failure modes ---

print("""
EXERCISE: For each failure mode you identified, propose a specific fix.

Example format:
- Failure Mode: AMNESIA (Safety-Critical) -- allergy forgotten in Session 3
  Root Cause: Sliding window dropped Session 1 messages
  Proposed Fix: Entity store with importance=1.0 guarantees allergy persistence

YOUR ANALYSIS:
1. Top failure mode from baseline:
   Root cause:
   Proposed fix:

2. Top failure mode from hybrid system (if any):
   Root cause:
   Proposed fix:

3. Which failure mode is most dangerous in a clinical setting? Why?

""")

---

## 3.8 Scalability and Deployment Considerations

In [ ]:
# --- Latency Profiling ---

def profile_pipeline(hm, query, session_id, n_runs=10):
    """Profile each stage of the hybrid memory pipeline."""
    timings = defaultdict(list)

    for _ in range(n_runs):
        # Entity lookup
        t0 = time.perf_counter()
        safety = hm.entity_store.get_safety_critical()
        entity_ctx = hm.entity_store.format_for_context()
        timings["entity_lookup"].append((time.perf_counter() - t0) * 1000)

        # Embedding
        t0 = time.perf_counter()
        q_emb = embed_text(query)
        timings["embedding"].append((time.perf_counter() - t0) * 1000)

        # Vector retrieval
        t0 = time.perf_counter()
        results = hm.vector_store.retrieve(query)
        timings["vector_retrieval"].append((time.perf_counter() - t0) * 1000)

        # Scoring
        t0 = time.perf_counter()
        if results:
            for r in results:
                hm.compute_hybrid_score(r.get("score", 0.5), 30, r.get("metadata", {}).get("importance", 0.0))
        timings["scoring"].append((time.perf_counter() - t0) * 1000)

        # Context assembly
        t0 = time.perf_counter()
        context = hm.build_context(query, session_id)
        timings["full_context_assembly"].append((time.perf_counter() - t0) * 1000)

    print(f"{'Stage':<25} {'Mean (ms)':>10} {'P95 (ms)':>10}")
    print("-" * 47)
    for stage, times in timings.items():
        mean_t = np.mean(times)
        p95_t = np.percentile(times, 95)
        print(f"{stage:<25} {mean_t:>10.2f} {p95_t:>10.2f}")

    total_mean = sum(np.mean(t) for t in timings.values())
    print(f"\n{'TOTAL (excl. LLM)':<25} {total_mean:>10.2f} ms")
    print(f"{'LLM budget':<25} {'~4500':>10} ms")
    print(f"{'Est. total':<25} {total_mean + 4500:>10.0f} ms")

    return timings


# Run profiling (use the hybrid memory system from evaluation)
# First, rebuild a memory system with data
hm_profile = HybridMemorySystem(
    embedding_fn=embed_text,
    window_size=5,
    vector_top_k=5,
    max_context_tokens=4000
)

# Populate with a single patient's data
pid = patients[0].patient_id
sessions_for_patient = [s for s in all_sessions if s.patient_id == pid]
for session in sessions_for_patient:
    text_parts = []
    for msg in session.messages:
        hm_profile.add_message(msg.role, msg.content, session.session_id, msg.timestamp)
        text_parts.append(f"{msg.role}: {msg.content}")
    hm_profile.end_session(session.session_id, " ".join(text_parts))

print("Latency Profile (10 runs):")
print("=" * 47)
timings = profile_pipeline(hm_profile, "Does this patient have any drug allergies?", 5)

In [ ]:
# --- Scalability Analysis ---

print("""
EXERCISE: Scalability Questions

1. If the vector store grows to 1 million memories (from current ~5.2M across
   all patients), how would retrieval latency change with IndexFlatIP?
   What index type would you recommend switching to?

   HINT: IndexFlatIP is O(N) exact search. Consider:
   - IndexIVFFlat (inverted file index) for approximate search
   - IndexIVFPQ (product quantization) for memory-efficient approximate search
   - HNSW for low-latency approximate search

2. With 1,800 physician users and average 20 queries per patient visit,
   how many queries per second must the system handle at peak?
   Assume 60% of visits happen between 9 AM and 12 PM.

3. How would you implement per-patient FAISS index partitioning?
   What are the tradeoffs vs. a single global index?

YOUR ANSWERS:
1.

2.

3.
""")

---

## 3.9 Ethical and Regulatory Analysis

In [ ]:
# --- HIPAA Audit Logging Schema ---

print("""
EXERCISE: Design an Audit Logging Schema

HIPAA requires that all access to Protected Health Information (PHI) be logged.
The hybrid memory system accesses PHI on every query.

Design a log schema with the following requirements:
- Every memory read, write, and delete must be logged
- Logs must be immutable (append-only)
- Logs must be retained for 7 years
- Each log entry must be sufficient for a compliance audit

Required fields for each log entry:
1. timestamp (ISO 8601)
2. event_type (read | write | delete | search)
3. physician_id
4. patient_id
5. session_id
6. query_text_hash (SHA-256 of query, NOT the raw text)
7. memories_accessed (list of memory IDs)
8. memory_subsystem (entity_store | vector_store | summary | window)
9. response_summary_hash
10. ip_address
11. access_justification (treating_physician | emergency | audit_review)

YOUR SCHEMA (as a Python dataclass):
""")


@dataclass
class AuditLogEntry:
    """HIPAA-compliant audit log entry for memory access.

    TODO: Define all fields with appropriate types.
    Consider: which fields should be required vs optional?
    Which fields should be hashed for privacy?
    """
    # TODO: Implement
    pass


print("Implement the AuditLogEntry dataclass above.")

In [ ]:
# --- Entity Extraction Differential Analysis ---

print("Testing entity extraction across medical specialties...")

cardiology_text = (
    "Patient diagnosed with atrial fibrillation. Started warfarin 5mg daily. "
    "INR came back at 2.3. Target therapeutic range is 2.0 to 3.0. "
    "Also has allergy to aspirin. BNP was 450 pg/mL indicating heart failure."
)

endocrinology_text = (
    "Patient diagnosed with hypothyroidism. Started levothyroxine 50mcg daily. "
    "TSH came back at 8.2 mIU/L, above normal range. A1C is 6.8 percent. "
    "Patient has allergy to contrast dye."
)

oncology_text = (
    "Patient diagnosed with non-small cell lung cancer, stage IIIA. "
    "Started on carboplatin-based chemotherapy regimen. "
    "Creatinine came back at 1.5 mg/dL indicating renal impairment. "
    "Patient has allergy to sulfonamides. eGFR is 52 mL/min."
)

es_test = EntityStore()
for specialty, text in [("Cardiology", cardiology_text),
                         ("Endocrinology", endocrinology_text),
                         ("Oncology", oncology_text)]:
    entities = es_test.extract_entities(text, session_id=1, timestamp="2024-01-15T10:00:00")
    if entities is None:
        entities = []
    print(f"\n{specialty}: {len(entities)} entities extracted")
    for e in entities:
        print(f"  [{e.entity_type}] {e.name}: {e.value}")

print("""
\nEXERCISE: Ethical Impact Assessment (500 words)

Address the following questions:
(a) Who benefits from a clinical memory system?
    - Consider: physicians, patients, clinic administrators, insurers

(b) Who could be harmed?
    - Consider: patients with rare conditions, underserved populations,
      patients who want information forgotten

(c) What safeguards should be in place?
    - Consider: physician override, patient consent, audit trails,
      error reporting mechanisms

(d) Should this system be deployed without a physician always reviewing
    AI recommendations?
    - Consider: autonomy, liability, the difference between decision
      support and decision making

Write your assessment below:
""")

---

## Summary

Congratulations! You have built a complete hybrid memory system for clinical AI, implementing:

1. **Sliding Window Memory** -- for immediate conversational context
2. **Vector-Backed Long-Term Memory** -- for semantic retrieval of past conversations across sessions
3. **Entity Store** -- for structured, safety-critical clinical facts that must never be forgotten
4. **Running Summary** -- for compressed patient trajectory overview
5. **Hybrid Scoring** -- combining similarity, recency, and clinical importance
6. **Context Assembly** -- with XML-tagged sections and token budgeting

The key insight from this case study: **no single memory architecture is sufficient for safety-critical applications.** The sliding window baseline achieves low recall because it cannot look beyond the current session. Vector retrieval alone might miss safety-critical facts if the query is not semantically similar to the allergy mention. The entity store alone lacks the nuance of why decisions were made. Only by combining all four subsystems -- each compensating for the others' weaknesses -- do we achieve the reliability required for clinical deployment.

In production, this system would also need:
- HIPAA-compliant infrastructure (encryption, access controls, audit logging)
- Real-time safety validation (cross-referencing every recommendation against known allergies and contraindications)
- Memory consolidation for patients with long histories
- Drift monitoring to detect degradation over time
- A/B testing with clinical safety guardrails

The memory architecture techniques you learned here -- buffer, sliding window, summary, entity, vector, and hybrid -- apply far beyond healthcare. Any AI application that must maintain accurate state across sessions benefits from thoughtful memory design.